# PointNet++ Ortodontik Landmark - Colab GPU

Bu notebook mevcut PointNet++ modelini Google Colab Pro GPU uzerinde calistirir ve Average Localization Error (ALE) raporlar.

DiffusionNet sonuclari sabit tutulacaksa bu notebook PointNet++ karsilastirma kosusunu uretmek icindir.

In [ ]:
import os, platform, torch
print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 1. Drive bagla ve repoyu hazirla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!test -d comparative-study || git clone https://github.com/eckdev/comparative-study.git
%cd /content/comparative-study
!git pull
!pip install -q -r pointnet2_orthodontic_comparison/requirements.txt

## 2. Veri yollarini ayarla

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/orthodontic/data/dataset')
SPLITS_JSON = Path('/content/comparative-study/shared_splits/orthodontic_180_60_60_seed42.json')
TRANSFORM_DIR = Path('/content/drive/MyDrive/orthodontic/transforms/orthodontic_procrustes_rigid_20260627_143801')
RUN_ROOT = Path('/content/drive/MyDrive/orthodontic/pointnet2_runs')
USE_TRANSFORMS = TRANSFORM_DIR.exists()
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT exists:', DATA_ROOT.exists(), DATA_ROOT)
print('SPLITS_JSON exists:', SPLITS_JSON.exists(), SPLITS_JSON)
print('TRANSFORM_DIR exists:', TRANSFORM_DIR.exists(), TRANSFORM_DIR)
print('RUN_ROOT:', RUN_ROOT)

## 3. Dataset kontrolu

In [ ]:
import glob

ply_count = len(glob.glob(str(DATA_ROOT / 'Class*' / '*' / '*.ply')))
txt_count = len(glob.glob(str(DATA_ROOT / 'Class*' / 'Class*-Landmark' / '*' / '*.txt')))
print('PLY:', ply_count)
print('Landmark TXT:', txt_count)
assert DATA_ROOT.exists(), 'DATA_ROOT bulunamadi. Google Drive yolunu kontrol et.'
assert ply_count > 0 and txt_count > 0, 'Dataset eksik gorunuyor.'

## 4. Smoke test

In [ ]:
import shlex, subprocess

smoke_dir = RUN_ROOT / 'pointnet2_smoke_colab'
cmd = [
    'python', '-u', 'pointnet2_orthodontic_comparison/run_orthodontic_pointnet2.py',
    '--data-root', str(DATA_ROOT),
    '--splits-json', str(SPLITS_JSON),
    '--output-dir', str(smoke_dir),
    '--surface-points', '512',
    '--sa1-points', '128',
    '--sa2-points', '32',
    '--sa3-points', '8',
    '--nsample', '24',
    '--epochs', '2',
    '--patience', '2',
    '--batch-size', '2',
    '--target-mode', 'gaussian',
    '--heatmap-sigma', '3.5',
    '--use-normals',
    '--coord-weight', '0.25',
    '--postprocess', 'topk_softmax',
    '--topk', '10',
    '--device', 'auto',
]
if USE_TRANSFORMS:
    cmd.extend(['--transformation-dir', str(TRANSFORM_DIR)])
print(' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

## 5. Ana Colab Pro kosusu

In [ ]:
main_dir = RUN_ROOT / 'pointnet2_v2_gaussian_normals_p4096_e200'
cmd = [
    'python', '-u', 'pointnet2_orthodontic_comparison/run_orthodontic_pointnet2.py',
    '--data-root', str(DATA_ROOT),
    '--splits-json', str(SPLITS_JSON),
    '--output-dir', str(main_dir),
    '--surface-points', '4096',
    '--eval-surface-points', '4096',
    '--sa1-points', '1024',
    '--sa2-points', '256',
    '--sa3-points', '64',
    '--nsample', '32',
    '--epochs', '200',
    '--patience', '40',
    '--batch-size', '4',
    '--lr', '0.001',
    '--weight-decay', '0.0001',
    '--optimizer', 'adamw',
    '--scheduler', 'cosine',
    '--target-mode', 'gaussian',
    '--heatmap-sigma', '3.5',
    '--use-normals',
    '--coord-weight', '0.25',
    '--coord-temperature', '1.0',
    '--postprocess', 'topk_softmax',
    '--topk', '10',
    '--temperature', '1.0',
    '--device', 'auto',
]
if USE_TRANSFORMS:
    cmd.extend(['--transformation-dir', str(TRANSFORM_DIR)])
print(' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

## 6. A100 / yuksek VRAM buyuk kosu

In [ ]:
big_dir = RUN_ROOT / 'pointnet2_v2_gaussian_normals_p8192_e220'
big_cmd = [
    'python', '-u', 'pointnet2_orthodontic_comparison/run_orthodontic_pointnet2.py',
    '--data-root', str(DATA_ROOT),
    '--splits-json', str(SPLITS_JSON),
    '--output-dir', str(big_dir),
    '--surface-points', '8192',
    '--eval-surface-points', '8192',
    '--sa1-points', '2048',
    '--sa2-points', '512',
    '--sa3-points', '128',
    '--nsample', '32',
    '--epochs', '220',
    '--patience', '45',
    '--batch-size', '2',
    '--lr', '0.001',
    '--weight-decay', '0.0001',
    '--optimizer', 'adamw',
    '--scheduler', 'cosine',
    '--target-mode', 'gaussian',
    '--heatmap-sigma', '3.5',
    '--use-normals',
    '--coord-weight', '0.25',
    '--coord-temperature', '1.0',
    '--postprocess', 'topk_softmax',
    '--topk', '10',
    '--temperature', '1.0',
    '--device', 'auto',
]
if USE_TRANSFORMS:
    big_cmd.extend(['--transformation-dir', str(TRANSFORM_DIR)])
print('A100 / yuksek VRAM buyuk kosu basliyor:')
print(' '.join(shlex.quote(x) for x in big_cmd))
subprocess.run(big_cmd, check=True)

## 7. Evaluate-only postprocess denemeleri

In [ ]:
eval_base = main_dir
model_path = eval_base / 'best_model.pth'
eval_settings = [
    ('topk5_t1', 5, 1.0),
    ('topk10_t1', 10, 1.0),
    ('topk20_t1', 20, 1.0),
    ('topk10_t05', 10, 0.5),
]
for tag, topk, temp in eval_settings:
    out_dir = RUN_ROOT / f'pointnet2_eval_{tag}'
    eval_cmd = [
        'python', '-u', 'pointnet2_orthodontic_comparison/run_orthodontic_pointnet2.py',
        '--data-root', str(DATA_ROOT),
        '--splits-json', str(SPLITS_JSON),
        '--output-dir', str(out_dir),
        '--surface-points', '4096',
        '--eval-surface-points', '4096',
        '--sa1-points', '1024',
        '--sa2-points', '256',
        '--sa3-points', '64',
        '--nsample', '32',
        '--batch-size', '4',
        '--target-mode', 'gaussian',
        '--heatmap-sigma', '3.5',
        '--use-normals',
        '--coord-weight', '0.25',
        '--postprocess', 'topk_softmax',
        '--topk', str(topk),
        '--temperature', str(temp),
        '--device', 'auto',
        '--evaluate-only',
        '--model-path', str(model_path),
    ]
    if USE_TRANSFORMS:
        eval_cmd.extend(['--transformation-dir', str(TRANSFORM_DIR)])
    print(' '.join(shlex.quote(x) for x in eval_cmd))
    subprocess.run(eval_cmd, check=True)

## 8. Sonuclari ozetle

In [ ]:
import json

for metrics_path in sorted(RUN_ROOT.glob('pointnet2*/metrics*.json')):
    with open(metrics_path) as f:
        metrics = json.load(f)
    summary = metrics.get('pointnet2', {})
    if summary:
        print(metrics_path.parent.name, metrics_path.name, 'ALE=', round(summary.get('ale', -1), 4), 'median=', round(summary.get('median', -1), 4), 'max=', round(summary.get('max', -1), 4))